# AgentIR-style Qwen3-Embedding-0.6B on Modal

This notebook launches the pinned Tevatron/DeepSpeed training procedure on one Modal A100 80GB.
Training uses 5,238 DR-Synth WebShaper rows, 8,192-token reasoning/query inputs, and 4,096-token documents.
The model remains Qwen3-Embedding-0.6B. BrowseComp-Plus is evaluation-only; evaluation replays fixed agent searches.

**Start with the bounded 10-minute GPU benchmark.** CPU preparation and image building happen first and have separate costs.
All launch switches below default to off. Running the notebook top to bottom does not launch GPU work.

Read [MODAL_TRAINING_CHANGES.md](MODAL_TRAINING_CHANGES.md) for changes, limitations, and commands.
The old Colab experiment and its saved output are preserved in `browsecomp_dr_synth_trained_qwen06_replay_legacy.ipynb`.
Keep this notebook and the accompanying `agentir_*.py`, `modal_agentir.py`, and JSON configuration together.


## 1. Install the local launcher and authenticate

In a terminal in this folder:

```bash
python3 -m venv .venv_modal
source .venv_modal/bin/activate
python -m pip install -r requirements-modal.txt
python -m modal setup
```

The notebook's Python environment must have the same Modal SDK installed if launching from a cell.
All CUDA, Torch, Tevatron, and DeepSpeed dependencies are installed inside the Modal image.
The local machine does not need a GPU. Configure the Modal Workspace budget for your intended spending limit;
the scripts' cost estimates are not an account billing cap.


In [ ]:
from pathlib import Path
from dataclasses import asdict
import json, subprocess, sys
from agentir_config import TrainingConfig, training_steps, estimated_hourly_rate

PROJECT = Path.cwd()
required = ["modal_agentir.py", "agentir_config.py", "agentir_data.py", "agentir_training.py",
            "agentir_workload.py", "agentir_deepspeed.json"]
assert all((PROJECT / name).exists() for name in required), "Open the notebook from the project folder"
CONFIG = TrainingConfig()
MAX_QUERY_TOKENS = CONFIG.query_tokens          # 8192
MAX_PASSAGE_TOKENS = CONFIG.passage_tokens      # 4096
MAX_EVAL_QUERY_TOKENS = CONFIG.eval_query_tokens # 8192
print(json.dumps(asdict(CONFIG), indent=2))
print("Full training optimizer-step attempts:", training_steps())
print("Estimated A100 + 4 CPU cores + 32 GiB hourly cost:", round(estimated_hourly_rate(), 3))


## 2. Inspect the training recipe

The shared query/document encoder uses the official Tevatron contrastive loss, one positive and seven hard
negatives per query, plus in-batch negatives. Batch size is 4 with accumulation 2, two epochs, temperature 0.01,
learning rate 1e-4, LoRA r=16/alpha=64/dropout=0.1, zero weight decay, and DeepSpeed WarmupDecayLR with 10% warmup.
Gradient checkpointing is active in training mode. The custom sequential gradient-cache loop has been replaced.

Inputs longer than the query cap lose the beginning of the reasoning while retaining the full current query;
a query that cannot fit raises an error. Documents are right-truncated. This query-preservation policy is an
explicit adaptation from the original Tevatron collator. See the change log for source commit pins.

Edit `agentir_config.py` to change the experiment configuration. The new configuration, sources, package pins,
code hashes, and input policy determine a separate run ID. Old Colab checkpoints are not resumed.


In [ ]:
print(json.dumps(json.loads((PROJECT / "agentir_deepspeed.json").read_text()), indent=2))


## 3. Run the bounded benchmark

CPU preparation downloads the pinned model/training data and small corpus/trajectory samples to the persistent
Modal Volume. The GPU function has a 600-second timeout and no automatic retries. The first training accumulation
window uses synthetic full-length inputs to check memory at 8,192/4,096 tokens; these and warmup steps are excluded
from steady-state throughput. Benchmark weights are never used as a final baseline.

The benchmark then times document/query embedding and exact FAISS search over 100,195 synthetic document vectors.
Synthetic vectors measure full-index search cost; no relevance scores are reported. Samples are limited and the
projection is an estimate. If setup or long steps exhaust the budget, the report remains incomplete and no full
cost prediction is produced.

Equivalent terminal command: `python -m modal run --detach modal_agentir.py --mode benchmark`

Launches use `--detach`. Once the cloud `pipeline` is running, you may close the laptop.
Preparation and GPU dispatch both continue on Modal. Reports remain in the Volume;
the local JSON mirror requires this notebook process to stay connected until completion.


In [ ]:
RUN_BENCHMARK = False  # Set True only when ready to use Modal compute.
if RUN_BENCHMARK:
    subprocess.run([sys.executable, "-m", "modal", "run", "--detach", str(PROJECT / "modal_agentir.py"),
                    "--mode", "benchmark"], check=True, cwd=PROJECT)
else:
    print("Benchmark is ready. Set RUN_BENCHMARK=True or use the terminal command above.")


In [ ]:
reports = sorted((PROJECT / "modal_results").glob("*_benchmark_*.json"))
BENCHMARK_REPORT = json.loads(reports[-1].read_text()) if reports else None
if BENCHMARK_REPORT:
    print("Run ID:", BENCHMARK_REPORT["run_id"])
    print("Status:", BENCHMARK_REPORT["status"])
    print(json.dumps(BENCHMARK_REPORT.get("projection", {"message": "Projection unavailable; inspect incomplete stages"}), indent=2))
else:
    print("No benchmark has been run yet.")


## 4. Run full stages after reviewing the benchmark

Full stages are explicit and resumable. Each invocation accepts 120–25,200 seconds (up to 7 hours; default 1,800).
A complete benchmark is required; the launcher rejects projections above $40 and reserves an estimated budget
before each stage. These estimates exclude account-wide usage and do not replace Modal's billing budget.

Run `train` until it reports `complete`, then `encode` until complete, then `replay` until complete.
The corpus and replay data download on CPU before the relevant GPU phase. Checkpoints, corpus chunks, and search
batches persist in the `ir646-agentir` Volume. Never run benchmark/full stages concurrently.

For the October 30 FP16 loss-scale failure, do not restart from pretrained weights. Prepare a child run with
`--mode prepare` (CPU only), then run `agentir_resume.py` with the failed run ID as `--parent-run-id` and the
prepared run ID as `--child-run-id`. It verifies and copies the saved checkpoint and inherits the old benchmark
projection before training; no new GPU benchmark is needed. The saved optimizer state restores its own loss-scaler
minimum, so the training worker now applies the configured child floor after checkpoint loading. The integer JSON
floor of 0 maps to 1/1024 at runtime; confirm this in `loss_scaler_at_start` in the training report.

Bounded FP32/BF16 diagnostics are also available: prepare with `--precision fp32` or `--precision bf16`, use
`agentir_resume.py::warmstart_precision` to restore only the saved LoRA adapter (fresh optimizer/scheduler; not an exact
resume), then run `--mode precision-trial` for at most three optimizer attempts. This leaves the original checkpoint unchanged.
The BF16 check completed three steps without skips at about 16 seconds/step after startup, with 20.7 GiB peak allocation.
A bounded continuation can copy its complete step-3 BF16 checkpoint (including optimizer state) into a compatible child
and test up to 25 more steps. It continues the BF16 diagnostic from step 3; it does not restore the original FP16 optimizer.
That continuation completed 25 steps (global step 28) with no skipped updates, saved its checkpoint, and peaked at 20.9 GiB.

```bash
python -m modal run --detach modal_agentir.py --mode train --run-id RUN_ID --seconds 25200
python -m modal run --detach modal_agentir.py --mode encode --run-id RUN_ID --seconds 1800
python -m modal run --detach modal_agentir.py --mode replay --run-id RUN_ID --seconds 1800
```


In [ ]:
RUN_STAGE = None  # Choose "train", "encode", or "replay" after reviewing benchmark results.
RUN_ID = BENCHMARK_REPORT["run_id"] if BENCHMARK_REPORT else ""
STAGE_SECONDS = 25200 if RUN_STAGE == "train" else 1800
if RUN_STAGE is not None:
    assert BENCHMARK_REPORT and BENCHMARK_REPORT["status"] == "complete", "Finish and review the benchmark first"
    assert RUN_STAGE in {"train", "encode", "replay"}
    subprocess.run([sys.executable, "-m", "modal", "run", "--detach", str(PROJECT / "modal_agentir.py"),
                    "--mode", RUN_STAGE, "--run-id", RUN_ID, "--seconds", str(STAGE_SECONDS)],
                   check=True, cwd=PROJECT)


## 5. Results and interpretation

Completed replay exports per-question and per-agent CSVs with Evidence Recall@5, Evidence nDCG@10, Gold Recall@5,
and trajectory evidence union recall using the top five documents from each search. Per-search metrics first
average within each question, then across questions. Zero-search questions have zero union recall and no
per-search score. Partial replay cannot produce a full baseline report.

All 830 questions and six agents are retained, matching the existing preliminary baselines. This does **not**
implement the proposal's 100/730 development/test split or paired significance tests. Those remain evaluation
work before selecting a curation method. Frozen reasoning-plus-query baseline 3 is also still a separate task.
There is no live agent loop or answer-accuracy evaluation.

The Modal command returns a local JSON report in `modal_results/`. Detailed checkpoints and CSVs remain under
`/agentir/experiments/RUN_ID/` in the `ir646-agentir` Volume. Download a needed file with:

```bash
modal volume get ir646-agentir /agentir/experiments/RUN_ID/REMOTE_PATH ./LOCAL_FILENAME
```
